# Бустинги на широком наборе признаков

In [ ]:
'''
Что уже проверено на EDA (ноутбук _data_overview):
- данные чистые: нет пропусков и дублей, все товары есть в справочнике;
- 100 000 клиентов, каждый ровно в одном срезе; заказы каждого клиента обрезаны по его дате
  отсечения (scoring_date − 14 дней), история не длиннее двух лет;
- в выборку попадают клиенты с хотя бы одним заказом за последние 365 дней до отсечения;
- в идентификаторах и порядке строк утечек нет;
- формула n_orders / (recency + 30) без обучения даёт AP@20% 0.50 на train и 0.548 на Public,
  то есть тест ведёт себя как train;
- метки, посчитанные по истории на сдвинутых назад датах, ведут себя иначе, чем target,
  поэтому в обучение они не идут;
- в тесте больше новых клиентов и свежих заказов, чем в срезах train.

Что делаем здесь:
- широкий набор признаков: давность и частота по окнам, ритм заказов, чеки, скидки,
  лояльность, состав покупок;
- LightGBM и CatBoost с ручными параметрами;
- две схемы проверки: январь + февраль -> март и стратифицированная кросс-валидация;
- важность признаков и разбор верхушки рейтинга;
- отправка: смесь двух моделей, обученных на всём train.
'''
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("LAMODA_OUT_DIR", "."))

SEED = 42
GAP_DAYS = 14
N_FOLDS = 5
WINDOWS = [7, 14, 30, 60, 90, 180, 365]


def average_precision_at_20_percent(customer_id, target, score):
    ranking = (
        pd.DataFrame({"customer_id": np.asarray(customer_id).astype(str), "target": np.asarray(target), "score": np.asarray(score)})
        .sort_values(["score", "customer_id"], ascending=[False, True], kind="mergesort")
        .reset_index(drop=True)
    )
    k = int(np.ceil(len(ranking) * 0.20))
    top_target = ranking["target"].iloc[:k].to_numpy()
    precision_at_rank = np.cumsum(top_target) / np.arange(1, k + 1)
    denominator = min(k, int(ranking["target"].sum()))
    return (precision_at_rank * top_target).sum() / max(denominator, 1)


def ap_by_slice(meta, score):
    # AP@20% считается отдельно по каждому срезу: тест — это один срез
    out = {s: average_precision_at_20_percent(g["customer_id"], g["target"], score[g.index]) for s, g in meta.groupby("slice")}
    out["среднее"] = float(np.mean(list(out.values())))
    return out

## Данные

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["scoring_date"])
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
orders = pd.read_parquet(DATA_DIR / "orders.parquet")
products = pd.read_parquet(DATA_DIR / "products.parquet")
orders["order_date"] = pd.to_datetime(orders["order_date"])

cust = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
cust["slice"] = cust["scoring_date"].dt.strftime("%Y-%m-%d")
cust["cutoff"] = cust["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
assert cust["customer_id"].is_unique

lines = orders.merge(products, on="product_id", how="left", validate="m:1").merge(
    cust[["customer_id", "cutoff"]], on="customer_id", how="inner", validate="m:1"
)
lines["d"] = (lines["cutoff"] - lines["order_date"]).dt.days  # дней от заказа до отсечения клиента
assert len(lines) == len(orders)
assert lines["d"].between(0, 732).all(), "в истории есть заказы позже отсечения или старше двух лет"
print(cust.groupby(["part", "slice"])["target"].agg(["size", "mean"]))
print("строк в истории:", len(lines))

## Признаки

In [ ]:
def clean(name):
    return re.sub(r"\W+", "_", str(name)).strip("_").lower()


def build_features(lines):
    lines = lines.copy()
    lines["has_disc"] = (lines["discount_percent"] > 0).astype("float32")
    lines["has_loy"] = (lines["loyalty_percent"] > 0).astype("float32")
    lines["weekend"] = (lines["order_date"].dt.dayofweek >= 5).astype("float32")

    # --- уровень заказа, самые свежие заказы идут первыми
    ot = (
        lines.groupby(["customer_id", "order_id"], sort=False)
        .agg(
            d=("d", "first"),
            order_sum=("paid_price", "sum"),
            n_items=("paid_price", "size"),
            disc=("discount_percent", "mean"),
            loy=("loyalty_percent", "max"),
            weekend=("weekend", "first"),
        )
        .reset_index()
        .sort_values(["customer_id", "d", "order_id"])
        .reset_index(drop=True)
    )
    cid = ot["customer_id"]
    f = ot.groupby("customer_id").agg(
        recency=("d", "min"),
        tenure=("d", "max"),
        d_mean=("d", "mean"),
        d_std=("d", "std"),
        n_orders=("order_id", "size"),
        n_items=("n_items", "sum"),
        sum_paid=("order_sum", "sum"),
        order_sum_mean=("order_sum", "mean"),
        order_sum_median=("order_sum", "median"),
        order_sum_max=("order_sum", "max"),
        order_sum_std=("order_sum", "std"),
        items_per_order=("n_items", "mean"),
        items_max=("n_items", "max"),
        order_disc_mean=("disc", "mean"),
        order_loy_mean=("loy", "mean"),
        order_loy_max=("loy", "max"),
        weekend_share=("weekend", "mean"),
    )

    last = ot.drop_duplicates("customer_id").set_index("customer_id")
    first = ot.drop_duplicates("customer_id", keep="last").set_index("customer_id")
    f["last_order_sum"] = last["order_sum"]
    f["last_n_items"] = last["n_items"]
    f["last_disc"] = last["disc"]
    f["last_loy"] = last["loy"]
    f["first_loy"] = first["loy"]
    f["first_order_sum"] = first["order_sum"]
    f["loy_change"] = f["last_loy"] - f["first_loy"]
    f["last_sum_to_mean"] = f["last_order_sum"] / f["order_sum_mean"]

    # --- окна по времени
    for w in WINDOWS:
        m = ot["d"] <= w
        f[f"n_orders_{w}"] = m.groupby(cid).sum()
        f[f"sum_{w}"] = (ot["order_sum"] * m).groupby(cid).sum()
        f[f"n_items_{w}"] = (ot["n_items"] * m).groupby(cid).sum()
    for w in [90, 365]:
        m = ot["d"] <= w
        f[f"loy_max_{w}"] = ot["loy"].where(m).groupby(cid).max()
        f[f"disc_mean_{w}"] = ot["disc"].where(m).groupby(cid).mean()
        f[f"aov_{w}"] = f[f"sum_{w}"] / f[f"n_orders_{w}"].replace(0, np.nan)
    f["n_orders_older_365"] = f["n_orders"] - f["n_orders_365"]

    # --- уровень дня: ритм заказов
    days = ot.groupby(["customer_id", "d"], as_index=False).agg(day_orders=("order_id", "size"), day_sum=("order_sum", "sum"))
    days["gap"] = days.groupby("customer_id")["d"].diff()
    days["rk"] = days.groupby("customer_id").cumcount()
    gd = days.groupby("customer_id")
    f["n_days"] = gd.size()
    f["max_day_orders"] = gd["day_orders"].max()
    f["multi_order_days"] = (days["day_orders"] > 1).groupby(days["customer_id"]).sum()
    f = f.join(gd["gap"].agg(gap_mean="mean", gap_median="median", gap_std="std", gap_min="min", gap_max="max"))
    for k in [1, 2, 4, 9]:
        f[f"d_day_{k + 1}"] = days.loc[days["rk"] == k].set_index("customer_id")["d"]  # давность (k+1)-го с конца дня с заказом
    f["gap_last"] = days.loc[days["rk"] == 1].set_index("customer_id")["gap"]
    f["gap_prev"] = days.loc[days["rk"] == 2].set_index("customer_id")["gap"]
    for name, period, limit in [("months_12", 30, 12), ("months_24", 30, 25), ("weeks_13", 7, 13), ("weeks_26", 7, 26)]:
        b = days.assign(b=days["d"] // period)
        f[f"active_{name}"] = b.loc[b["b"] < limit].drop_duplicates(["customer_id", "b"]).groupby("customer_id").size()
    f[[c for c in f.columns if c.startswith("active_")]] = f[[c for c in f.columns if c.startswith("active_")]].fillna(0)

    # --- производные
    f["formula"] = f["n_orders"] / (f["recency"] + 30)
    f["days_formula"] = f["n_days"] / (f["recency"] + 30)
    f["orders_per_30d"] = f["n_orders"] / (f["tenure"] + 30) * 30
    f["recency_to_gap_mean"] = f["recency"] / f["gap_mean"]
    f["recency_to_gap_median"] = f["recency"] / f["gap_median"]
    f["recency_minus_gap_max"] = f["recency"] - f["gap_max"]
    f["gap_cv"] = f["gap_std"] / f["gap_mean"]
    f["recency_to_tenure"] = f["recency"] / (f["tenure"] + 1)
    f["share_orders_30"] = f["n_orders_30"] / f["n_orders"]
    f["share_orders_90"] = f["n_orders_90"] / f["n_orders"]
    f["share_orders_365"] = f["n_orders_365"] / f["n_orders"]
    f["trend_90_vs_365"] = f["n_orders_90"] / (f["n_orders_365"] - f["n_orders_90"] + 1)
    f["trend_30_vs_90"] = f["n_orders_30"] / (f["n_orders_90"] - f["n_orders_30"] + 1)
    f["share_sum_90"] = f["sum_90"] / f["sum_paid"].replace(0, np.nan)

    # --- уровень позиции: цены, скидки, лояльность, состав покупок
    gl = lines.groupby("customer_id")
    f = f.join(
        gl.agg(
            price_mean=("paid_price", "mean"),
            price_median=("paid_price", "median"),
            price_max=("paid_price", "max"),
            price_min=("paid_price", "min"),
            disc_mean=("discount_percent", "mean"),
            disc_max=("discount_percent", "max"),
            disc_share=("has_disc", "mean"),
            loy_mean=("loyalty_percent", "mean"),
            loy_share=("has_loy", "mean"),
            n_loy_values=("loyalty_percent", "nunique"),
            n_products=("product_id", "nunique"),
            n_brands=("brand", "nunique"),
            n_cat=("category", "nunique"),
            n_cat2=("category2", "nunique"),
            n_gender=("gender_product", "nunique"),
        )
    )
    f["repeat_product_share"] = 1 - f["n_products"] / f["n_items"]
    f["brands_per_item"] = f["n_brands"] / f["n_items"]
    f["cat2_per_item"] = f["n_cat2"] / f["n_items"]

    for col, prefix in [("category", "cat"), ("gender_product", "gender")]:
        share = pd.crosstab(lines["customer_id"], lines[col], normalize="index")
        share.columns = [f"{prefix}_share_{clean(c)}" for c in share.columns]
        f = f.join(share)
        recent = lines.loc[lines["d"] <= 90]
        share = pd.crosstab(recent["customer_id"], recent[col], normalize="index")
        share.columns = [f"{prefix}_share90_{clean(c)}" for c in share.columns]
        f = f.join(share)

    for col in ["category2", "brand"]:
        top = (
            lines.groupby(["customer_id", col], observed=True).size().rename("n").reset_index()
            .sort_values(["customer_id", "n", col], ascending=[True, False, True])
            .drop_duplicates("customer_id").set_index("customer_id")
        )
        f[f"top_{col}_share"] = top["n"] / f["n_items"]
        if col == "category2":
            f["top_category2"] = top[col].astype(str)

    return f.replace([np.inf, -np.inf], np.nan)


feats = build_features(lines)
data = cust.merge(feats, left_on="customer_id", right_index=True, how="left", validate="1:1")
assert data["recency"].notna().all() and len(data) == len(cust)

CAT_COLS = ["top_category2"]
data[CAT_COLS] = data[CAT_COLS].astype("category")
FEATURES = list(feats.columns)
print("признаков:", len(FEATURES))

tr = data[data["part"].eq("train")].reset_index(drop=True)
te = data[data["part"].eq("test")].set_index("customer_id").loc[sample["customer_id"]].reset_index()
y = tr["target"].astype(int)
data.drop(columns=["cutoff"]).to_parquet(OUT_DIR / "features.parquet", index=False)

## Модели

In [ ]:
LGB_PARAMS = dict(
    objective="binary", learning_rate=0.03, num_leaves=31, min_child_samples=100,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=5.0, cat_smooth=50,
    n_estimators=4000, random_state=SEED, n_jobs=-1, verbose=-1,
)
CB_PARAMS = dict(
    loss_function="Logloss", learning_rate=0.05, depth=6, l2_leaf_reg=5.0,
    iterations=4000, random_seed=SEED, thread_count=-1, verbose=0,
)
EARLY_STOP = 200


def cb_frame(X):
    X = X.copy()
    X[CAT_COLS] = X[CAT_COLS].astype(str)
    return X


def fit_lgb(X_tr, y_tr, X_va=None, y_va=None, n_iter=None):
    params = dict(LGB_PARAMS, n_estimators=n_iter or LGB_PARAMS["n_estimators"])
    model = lgb.LGBMClassifier(**params)
    if X_va is None:
        model.fit(X_tr, y_tr)
    else:
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="binary_logloss",
                  callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    importance = pd.Series(model.booster_.feature_importance("gain"), index=FEATURES)
    return model, lambda X: model.predict_proba(X)[:, 1], model.best_iteration_ or params["n_estimators"], importance / importance.sum()


def fit_cb(X_tr, y_tr, X_va=None, y_va=None, n_iter=None):
    params = dict(CB_PARAMS, iterations=n_iter or CB_PARAMS["iterations"])
    model = CatBoostClassifier(**params, cat_features=CAT_COLS)
    if X_va is None:
        model.fit(cb_frame(X_tr), y_tr)
    else:
        model.fit(cb_frame(X_tr), y_tr, eval_set=(cb_frame(X_va), y_va), early_stopping_rounds=EARLY_STOP)
    importance = pd.Series(model.get_feature_importance(), index=FEATURES)
    best = model.get_best_iteration()
    return model, lambda X: model.predict_proba(cb_frame(X))[:, 1], (best + 1) if best is not None else params["iterations"], importance / importance.sum()


MODELS = {"lightgbm": fit_lgb, "catboost": fit_cb}


def rank_blend(preds):
    return np.mean([pd.Series(p).rank(pct=True).to_numpy() for p in preds], axis=0)

## Проверка: январь + февраль -> март

In [ ]:
slices_sorted = sorted(tr["slice"].unique())
is_va = tr["slice"].eq(slices_sorted[-1]).to_numpy()
X_all = tr[FEATURES]

time_pred = {}
rows = [{"модель": "формула", "AP@20% март": average_precision_at_20_percent(tr.loc[is_va, "customer_id"], y[is_va], tr.loc[is_va, "formula"])}]
for name, fit in MODELS.items():
    _, predict, best_iter, _ = fit(X_all[~is_va], y[~is_va], X_all[is_va], y[is_va])
    time_pred[name] = predict(X_all[is_va])
    rows.append({"модель": name, "AP@20% март": average_precision_at_20_percent(tr.loc[is_va, "customer_id"], y[is_va], time_pred[name]), "итераций": best_iter})
rows.append({"модель": "смесь", "AP@20% март": average_precision_at_20_percent(tr.loc[is_va, "customer_id"], y[is_va], rank_blend(list(time_pred.values())))})
display(pd.DataFrame(rows).set_index("модель"))

## Проверка: кросс-валидация

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
strata = tr["slice"] + "_" + y.astype(str)

oof = {name: np.zeros(len(tr)) for name in MODELS}
best_iters = {name: [] for name in MODELS}
importances = {name: [] for name in MODELS}
fold_scores = []
for fold, (i_tr, i_va) in enumerate(skf.split(X_all, strata)):
    row = {"fold": fold}
    for name, fit in MODELS.items():
        _, predict, best_iter, imp = fit(X_all.iloc[i_tr], y.iloc[i_tr], X_all.iloc[i_va], y.iloc[i_va])
        oof[name][i_va] = predict(X_all.iloc[i_va])
        best_iters[name].append(best_iter)
        importances[name].append(imp)
        row[name] = average_precision_at_20_percent(tr["customer_id"].iloc[i_va], y.iloc[i_va], oof[name][i_va])
    fold_scores.append(row)
print("AP@20% по фолдам (все срезы вместе):")
display(pd.DataFrame(fold_scores).set_index("fold"))

oof["смесь"] = rank_blend([oof[name] for name in MODELS])
cv_table = pd.DataFrame(
    {"формула": ap_by_slice(tr, tr["formula"].to_numpy()), **{name: ap_by_slice(tr, p) for name, p in oof.items()}}
).T
print("AP@20% по срезам на прогнозах вне фолда:")
display(cv_table)
print("Итераций до остановки:", {k: [int(i) for i in v] for k, v in best_iters.items()})
print("Корреляция прогнозов моделей (Спирмен):", round(float(pd.Series(oof["lightgbm"]).corr(pd.Series(oof["catboost"]), method="spearman")), 4))

## Важность признаков и верхушка рейтинга

In [ ]:
imp = pd.DataFrame({name: pd.concat(v, axis=1).mean(axis=1) for name, v in importances.items()})
imp["среднее"] = imp.mean(axis=1)
print("Доля важности, топ-40:")
display(imp.sort_values("среднее", ascending=False).head(40))
print("Признаки с нулевой важностью:", imp.index[imp["среднее"] == 0].tolist())

res = tr[["customer_id", "slice", "target", "recency", "n_orders"]].copy()
res["score"] = oof["смесь"]
res["pct"] = res.groupby("slice")["score"].rank(pct=True, ascending=False)
res["bucket"] = pd.cut(res["pct"], [0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 1.0])
print("Доля target = 1 по положению в рейтинге (доля сверху):")
display(res.groupby("bucket", observed=True)["target"].agg(["mean", "size"]).T)

top = res[res["pct"] <= 0.20]
res["n_bin"] = pd.cut(res["n_orders"], [0, 1, 2, 3, 5, 9, 20, 10_000], labels=["1", "2", "3", "4-5", "6-9", "10-20", "21+"])
res["in_top"] = res["pct"] <= 0.20
print("Верхние 20% рейтинга по числу заказов: доля группы в топе, точность в топе, доля покупателей вне топа")
display(
    pd.DataFrame(
        {
            "клиентов": res.groupby("n_bin", observed=True).size(),
            "доля в топ-20%": res.groupby("n_bin", observed=True)["in_top"].mean(),
            "target в топе": res[res["in_top"]].groupby("n_bin", observed=True)["target"].mean(),
            "target вне топа": res[~res["in_top"]].groupby("n_bin", observed=True)["target"].mean(),
        }
    )
)

## Отправка

In [ ]:
test_pred = {}
for name, fit in MODELS.items():
    n_iter = int(np.mean(best_iters[name]) * 1.15)
    _, predict, _, _ = fit(X_all, y, n_iter=n_iter)
    test_pred[name] = predict(te[FEATURES])
    print(name, "итераций:", n_iter)

submission = pd.DataFrame({"customer_id": te["customer_id"].to_numpy(), "score": rank_blend(list(test_pred.values()))})
assert submission["customer_id"].tolist() == sample["customer_id"].tolist()
assert submission["score"].between(0, 1).all()
submission.to_csv(OUT_DIR / "submission.csv", index=False)

pd.DataFrame({"customer_id": tr["customer_id"], "slice": tr["slice"], "target": y, **oof}).to_parquet(OUT_DIR / "oof.parquet", index=False)
pd.DataFrame({"customer_id": te["customer_id"], **test_pred}).to_parquet(OUT_DIR / "test_pred.parquet", index=False)
print("Корреляция прогнозов на тесте (Спирмен):",
      round(float(pd.Series(test_pred["lightgbm"]).corr(pd.Series(test_pred["catboost"]), method="spearman")), 4))
display(submission.describe().T)